We start by exploring ways to improve our model training.
Part 1: Initialization.
First our model had a very high loss, when the $P = ((1)_{i,j})$ model would have around 3-ish loss, making us lose a lot of compute to reach such levels.
To circumvent that, we set the bias of output layer to zero, and scale down the weights of that layer by 100.

Second, there's a high number of saturated neurones (neurones with instances where the grad update for some but not all weights is 0) -- thankfully no dead neurones tho, which are neurones whose grad = 0 for all inputs, meaning that `p += - k * p.grad` wouldn't really affect them and they never update. This is due to the fact that the values reaching our $tanh$ we so low/great they turned into $1$s or $-1$s, making partial derivatives for our weights $0$. to circumvent that, we scalled down our hiden layer's weights.

These two led to decreasing our loss significantly.

Next step is how to more structurally treat the previous scalin-downs?
Kaiming initializations is basically the factor (for tanh for example is 5/3) times 1/square root of fan_in (fan in is the max number of inputs).

### newer methods
Currently, initialisation isn't that critical because of new developed methods that reduce the need for a perfect initialization.
An example is the **batch normalization** which we're going to implement.

Right before reaching the squaching element (say tanh in our case) -- or commonly after layers that have a multiplication, we standarize our matrix (i.e. make it (0 mean ,1 std) distributed) by substracting the mean and dividing by the standard dev. 
However, and to leave room for the model to 'wiggle around' and play with it as it sees fit, we add one more weight and bias to let it manipulate the standarized matrix (i.e. $w \cdot \frac{A - \mu_A }{\sigma_A} + b$). It's worth noting that since we're substracting the mean, the bias of the multiplication layer becomes useless, and so it can be removed, the bias of our normalization layer fills its role.

A side effect of using batch normalisation is that, since we're normalizing with respect to matrix $A$ of a single batch each time, it kind of changes each time, like some sort of data augmentation (karpathy's words not mine) / overfit prevention effect, which is one of the reasons batch norm works so well.

Now when it comes to inference, remember that because of the added $\mu$ and $\sigma$, we will need to have values for these as well to be able to infer... There are two ways to do this, either you calculate the mean and standard div of the whole train set at the end and use them (strongly not advised bcz of computation cost) or you keep track of running avrages of mean and std (`avg = 0.999*avg + 0.001* current_avg` for example), the results of this running average will be similar to what you'd get from averaging one the whole train set but with minimal computation.




And so the schema is typicall: weight layer, normalization layer, and a non linearity layer.

Finally we introduced some diagnostic tools to check if our training process is running smoothly: we check the distribution of weights for linear layers, the distribution of gradients for tanh layers, the weight gradients distributions and the ratios of p.grad to p, the goal is to not have very fat tails / squashed distributions. we also checked the effects of gain updates (say 5/3 for tanh for examples) on said distributions, as well as our initializations with 1/sqrt (fan in).

One thing to highlight is that the batch norm really helped us stabilize the model as previously mentioned distributions, but it's still good to have a nice initialization and gain updates. For learning ratio (`lr * p.grad.std / p.std`) we typically want it to be around `10**-3` and so it's good to monitor and change `lr` accordingly to accomodate that.